In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# PARAMETERS
CHUNK_SIZE = 500
STEP_SIZE = 200
EPOCHS = 50
LR = 0.000267
BATCH_SIZE = 64
VAL_SIZE = 0.2
PATIENCE = 5


### Предобработка данных

In [ ]:
# Чтение файла
def read_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return f.read()

# Разбиение текста на отрывки
def chunk_text(text, chunk_size=200, step=200):
    tokens = text.split()
    chunks = []
    start = 0
    while start < len(tokens):
        chunk = tokens[start:start+chunk_size]
        if not chunk:
            break
        chunks.append(" ".join(chunk))
        start += step
    return chunks

# Функция для подготовки фиксированных чанков текста от разных авторов
def prepare_author_chunks(author_files, chunk_size=200, step=200):
    texts, labels, label2author = [], [], {}
    # Перебираем все файлы, соответствующие авторам
    for label_idx, filepath in enumerate(author_files):
        # Извлекаем имя автора из имени файла
        author_name = os.path.splitext(os.path.basename(filepath))[0]
        label2author[label_idx] = author_name
        # Разбиваем текст автора на чанки фиксированного размера и шага
        chunks = chunk_text(read_file(filepath), chunk_size, step)
        # Добавляем полученные чанки в общий список текстов
        texts.extend(chunks)
        # Добавляем соответствующую метку (label_idx) для каждого чанка
        labels.extend([label_idx] * len(chunks))

    # Возвращаем список текстов, список меток и отображение меток в имена авторов
    return texts, labels, label2author

### Классы датасета и персептрона

In [3]:
class TextDataset(Dataset):
    def __init__(self, X_vectors, y_labels):
        if hasattr(X_vectors, 'toarray'):
            X_vectors = X_vectors.toarray()
        self.X = torch.tensor(X_vectors, dtype=torch.float32)
        self.y = torch.tensor(y_labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class DeepMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim1=100, hidden_dim2=50, output_dim=6, dropout_rate=0.54):
        super(DeepMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.fc3 = nn.Linear(hidden_dim2, output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return x


### Функции для обучения и предсказания

In [4]:
def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3, device='cpu', patience=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
    
    best_val_loss = float('inf')
    best_state_dict = None
    epochs_no_improve = 0

    for epoch in range(1, epochs+1):
        model.train()
        total_train_loss = 0
        
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        
        avg_train_loss = total_train_loss / len(train_loader)
        
        model.eval()
        val_loss, all_preds, all_targets = 0, [], []
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                outputs = model(X)
                loss = criterion(outputs, y)
                val_loss += loss.item()
                preds = torch.argmax(outputs, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(y.cpu().numpy())
        avg_val_loss = val_loss / len(val_loader)
        scheduler.step(avg_val_loss)
        val_acc = accuracy_score(all_targets, all_preds)
        
        print(f"Epoch [{epoch}/{epochs}]: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}, Val Acc = {val_acc:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state_dict = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered.")
                break

    if best_state_dict:
        model.load_state_dict(best_state_dict)
    return model

def predict_text(model, vectorizer, text, chunk_size=200, step=200, device='cpu'):
    model.eval()
    chunks = chunk_text(text, chunk_size, step)
    if not chunks:
        return 0
    X = vectorizer.transform(chunks)
    X_torch = torch.tensor(X.toarray(), dtype=torch.float32).to(device)
    
    with torch.no_grad():
        logits = model(X_torch)
        probabilities = torch.softmax(logits, dim=1)
        avg_probabilities = probabilities.mean(dim=0)
        return torch.argmax(avg_probabilities).item()


### Подготовка данных: загрузка текстов, нарезка чанков, обучение модели

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

author_files = ["Fry.txt", "Genri.txt", "Simak.txt", "Bulgakov.txt", "Bradbury.txt", "Strugatskie.txt"]
path_to_files = "texts/"
full_paths = [path_to_files + filename for filename in author_files]
texts, labels, label2author = prepare_author_chunks(full_paths, CHUNK_SIZE, STEP_SIZE)
num_classes = len(label2author)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=VAL_SIZE, random_state=42, stratify=labels)
print(f"Train chunks: {len(train_texts)}, Val chunks: {len(val_texts)}")

vectorizer = TfidfVectorizer(max_features=8000, ngram_range=(1,2), analyzer='word', token_pattern=r'\w+')
X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)

train_dataset = TextDataset(X_train, train_labels)
val_dataset = TextDataset(X_val, val_labels)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

input_dim = X_train.shape[1]
model = DeepMLP(input_dim, hidden_dim1=93, hidden_dim2=65, output_dim=num_classes, dropout_rate=0.2).to(device)

print("\n--- Training the improved model ---")
model = train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, device=device, patience=PATIENCE)

model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for X, y in val_loader:
        X, y = X.to(device), y.to(device)
        preds = torch.argmax(model(X), dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(y.cpu().numpy())
val_acc = accuracy_score(all_true, all_preds)
print("\n=== Final validation evaluation ===")
print(f"Val Accuracy: {val_acc:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(all_true, all_preds))
print("\nClassification Report:")
print(classification_report(all_true, all_preds, target_names=[label2author[i] for i in range(num_classes)]))


Using device: cuda
Train chunks: 7350, Val chunks: 1838

--- Training the improved model ---


/home/aqqq/venvs/venv1/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/50]: Train Loss = 1.6823, Val Loss = 1.4534, Val Acc = 0.3542
Epoch [2/50]: Train Loss = 1.1320, Val Loss = 0.8451, Val Acc = 0.8205
Epoch [3/50]: Train Loss = 0.5974, Val Loss = 0.3955, Val Acc = 0.9489
Epoch [4/50]: Train Loss = 0.2882, Val Loss = 0.1984, Val Acc = 0.9869
Epoch [5/50]: Train Loss = 0.1499, Val Loss = 0.1095, Val Acc = 0.9918
Epoch [6/50]: Train Loss = 0.0859, Val Loss = 0.0680, Val Acc = 0.9935
Epoch [7/50]: Train Loss = 0.0567, Val Loss = 0.0485, Val Acc = 0.9946
Epoch [8/50]: Train Loss = 0.0385, Val Loss = 0.0378, Val Acc = 0.9956
Epoch [9/50]: Train Loss = 0.0295, Val Loss = 0.0296, Val Acc = 0.9978
Epoch [10/50]: Train Loss = 0.0237, Val Loss = 0.0247, Val Acc = 0.9978
Epoch [11/50]: Train Loss = 0.0190, Val Loss = 0.0221, Val Acc = 0.9967
Epoch [12/50]: Train Loss = 0.0166, Val Loss = 0.0191, Val Acc = 0.9978
Epoch [13/50]: Train Loss = 0.0154, Val Loss = 0.0174, Val Acc = 0.9984
Epoch [14/50]: Train Loss = 0.0121, Val Loss = 0.0161, Val Acc = 0.9984
E

### Тестирование

In [6]:
df = pd.read_csv("author_classification.csv")
test_true = []
test_pred = []

test_files_dir = "texts/"
print("\n--- Classifying test files using the improved model ---")
for _, row in df.iterrows():
    fname = os.path.join(test_files_dir, row['filename'])
    true_author = row['author']
    
    if not os.path.exists(fname):
        print(f"File {fname} not found, skipping...")
        continue
    
    text = read_file(fname)
    pred_label = predict_text(model, vectorizer, text, chunk_size=CHUNK_SIZE, step=STEP_SIZE, device=device)
    pred_author = label2author[pred_label]
    
    test_true.append(true_author)
    test_pred.append(pred_author)
    
    print(f"File {fname} -> Predicted: {pred_author} (True: {true_author})")

print("\n=== Evaluation on test set ===")
print("Accuracy:", accuracy_score(test_true, test_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(test_true, test_pred, labels=list(label2author.values())))
print("\nClassification Report:")
print(classification_report(test_true, test_pred, labels=list(label2author.values())))



--- Classifying test files using the improved model ---
File texts/author1.txt -> Predicted: Genri (True: Genri)
File texts/author2.txt -> Predicted: Simak (True: Simak)
File texts/author3.txt -> Predicted: Genri (True: Genri)
File texts/author4.txt -> Predicted: Bulgakov (True: Bulgakov)
File texts/author5.txt -> Predicted: Genri (True: Genri)
File texts/author6.txt -> Predicted: Bradbury (True: Bradbury)
File texts/author7.txt -> Predicted: Fry (True: Fry)
File texts/author8.txt -> Predicted: Fry (True: Fry)
File texts/author9.txt -> Predicted: Strugatskie (True: Strugatskie)
File texts/author10.txt -> Predicted: Fry (True: Bradbury)
File texts/author11.txt -> Predicted: Bulgakov (True: Bulgakov)
File texts/author12.txt -> Predicted: Fry (True: Bradbury)
File texts/author13.txt -> Predicted: Genri (True: Genri)
File texts/author14.txt -> Predicted: Bulgakov (True: Bulgakov)
File texts/author15.txt -> Predicted: Simak (True: Simak)
File texts/author16.txt -> Predicted: Simak (True: S